# Код курсовой работы на тему *Исследование и сравнительный анализ возможностей моделей BERT4Rec и SASRec в рекомендательных системах онлайн-кинотеатров*

## Загрузка библиотек

In [1]:
!pip install -q replay-rec[all]
!pip install wandb
!pip install -q rs-datasets

import wandb
import lightning as L
from lightning.pytorch.loggers import CSVLogger, WandbLogger
from lightning.pytorch.callbacks import ModelCheckpoint
from torch.utils.data import DataLoader
import torch

from replay.metrics import OfflineMetrics, Recall, Precision, MAP, NDCG, HitRate, MRR
from replay.metrics.torch_metrics_builder import metrics_to_df
from replay.splitters import LastNSplitter
from replay.utils import get_spark_session
from replay.data import (
    FeatureHint,
    FeatureInfo,
    FeatureSchema,
    FeatureSource,
    FeatureType,
    Dataset,
)
from replay.models.nn.optimizer_utils import FatOptimizerFactory
from replay.models.nn.sequential.callbacks import (
    ValidationMetricsCallback,
    SparkPredictionCallback,
    PandasPredictionCallback, 
    TorchPredictionCallback,
    QueryEmbeddingsPredictionCallback,
)
from replay.models.nn.sequential.postprocessors import RemoveSeenItems
from replay.data.nn import (
    SequenceTokenizer,
    SequentialDataset,
    TensorFeatureSource,
    TensorSchema,
    TensorFeatureInfo
)
from replay.models.nn.sequential import Bert4Rec
from replay.models.nn.sequential.bert4rec import (
    Bert4RecPredictionDataset,
    Bert4RecTrainingDataset,
    Bert4RecValidationDataset,
    Bert4RecPredictionBatch,
    Bert4RecModel
)
from replay.models.nn.sequential import SasRec
from replay.models.nn.sequential.sasrec import (
    SasRecPredictionDataset,
    SasRecTrainingDataset,
    SasRecValidationDataset,
    SasRecPredictionBatch,
    SasRecModel
)

import pandas as pd
from rs_datasets import MovieLens


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
beatrix-jupyterlab 2024.66.154055 requires jupyterlab~=3.6.0, but you have jupyterlab 4.2.5 which is incompatible.
distributed 2024.7.1 requires dask==2024.7.1, but you have dask 2024.9.1 which is incompatible.
jupyterlab 4.2.5 requires jupyter-lsp>=2.0.0, but you have jupyter-lsp 1.5.1 which is incompatible.
jupyterlab-lsp 5.1.0 requires jupyter-lsp>=2.0.0, but you have jupyter-lsp 1.5.1 which is incompatible.
rapids-dask-dependency 24.8.0a0 requires dask==2024.7.1, but you have dask 2024.9.1 which is incompatible.



## Подготовка данных

In [4]:
movielens = MovieLens("1m")
interactions = movielens.ratings
user_features = movielens.users
item_features = movielens.items

interactions["timestamp"] = interactions["timestamp"].astype("int64")
interactions = interactions.sort_values(by="timestamp")
interactions["timestamp"] = interactions.groupby("user_id").cumcount()
splitter = LastNSplitter(
    N=1,
    divide_column="user_id",
    query_column="user_id",
    strategy="interactions",
)

raw_test_events, raw_test_gt = splitter.split(interactions)
raw_validation_events, raw_validation_gt = splitter.split(raw_test_events)
raw_train_events = raw_validation_events

5.93MB [00:02, 2.88MB/s]                            


In [7]:
def prepare_feature_schema(is_ground_truth: bool) -> FeatureSchema:
    base_features = FeatureSchema(
        [
            FeatureInfo(
                column="user_id",
                feature_hint=FeatureHint.QUERY_ID,
                feature_type=FeatureType.CATEGORICAL,
            ),
            FeatureInfo(
                column="item_id",
                feature_hint=FeatureHint.ITEM_ID,
                feature_type=FeatureType.CATEGORICAL,
            ),
        ]
    )
    if is_ground_truth:
        return base_features

    all_features = base_features + FeatureSchema(
        [
            FeatureInfo(
                column="timestamp",
                feature_type=FeatureType.NUMERICAL,
                feature_hint=FeatureHint.TIMESTAMP,
            ),
        ]
    )
    return all_features

In [8]:
train_dataset = Dataset(
    feature_schema=prepare_feature_schema(is_ground_truth=False),
    interactions=raw_train_events,
    query_features=user_features,
    item_features=item_features,
    check_consistency=True,
    categorical_encoded=False,
)
validation_dataset = Dataset(
    feature_schema=prepare_feature_schema(is_ground_truth=False),
    interactions=raw_validation_events,
    query_features=user_features,
    item_features=item_features,
    check_consistency=True,
    categorical_encoded=False,
)
validation_gt = Dataset(
    feature_schema=prepare_feature_schema(is_ground_truth=True),
    interactions=raw_validation_gt,
    check_consistency=True,
    categorical_encoded=False,
)
test_dataset = Dataset(
    feature_schema=prepare_feature_schema(is_ground_truth=False),
    interactions=raw_test_events,
    query_features=user_features,
    item_features=item_features,
    check_consistency=True,
    categorical_encoded=False,
)
test_gt = Dataset(
    feature_schema=prepare_feature_schema(is_ground_truth=True),
    interactions=raw_test_gt,
    check_consistency=True,
    categorical_encoded=False,
)

In [11]:
ITEM_FEATURE_NAME = "item_id_seq"

tensor_schema = TensorSchema(
    TensorFeatureInfo(
        name=ITEM_FEATURE_NAME,
        is_seq=True,
        feature_type=FeatureType.CATEGORICAL,
        feature_sources=[TensorFeatureSource(FeatureSource.INTERACTIONS, train_dataset.feature_schema.item_id_column)],
        feature_hint=FeatureHint.ITEM_ID,
        embedding_dim=300,
    )
)

tokenizer = SequenceTokenizer(tensor_schema, allow_collect_to_master=True)
tokenizer.fit(train_dataset)

sequential_train_dataset = tokenizer.transform(train_dataset)

sequential_validation_dataset = tokenizer.transform(validation_dataset)
sequential_validation_gt = tokenizer.transform(validation_gt, [tensor_schema.item_id_feature_name])

sequential_validation_dataset, sequential_validation_gt = SequentialDataset.keep_common_query_ids(
    sequential_validation_dataset, sequential_validation_gt
)
test_query_ids = test_gt.query_ids
test_query_ids_np = tokenizer.query_id_encoder.transform(test_query_ids)["user_id"].values
sequential_test_dataset = tokenizer.transform(test_dataset).filter_by_query_id(test_query_ids_np)

## Подключение логирования при помощи Weights&Biases

In [15]:
!wandb login 8a20d5ca86ed554dbf4f3678d292cd38365f8b10

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


## Инициализация и обучение BERT4Rec

In [54]:
MAX_SEQ_LEN = 200
BATCH_SIZE = 512
NUM_WORKERS = 4

model = Bert4Rec(
    tensor_schema,
    block_count=2,
    head_count=4,
    max_seq_len=MAX_SEQ_LEN,
    hidden_size=300,
    dropout_rate=0.5,
    optimizer_factory=FatOptimizerFactory(learning_rate=0.001),
)
checkpoint_callback_bert = ModelCheckpoint(
    dirpath=".checkpoints_bert",
    save_top_k=1,
    verbose=True,
    # if you use multiple dataloaders, then add the serial number of the dataloader to the suffix of the metric name.
    # For example,"recall@10/dataloader_idx_0"
    monitor="recall@10",
    mode="max",
)

validation_metrics_callback = ValidationMetricsCallback(
    metrics=["mrr", "ndcg", "recall", "precision"],
    ks=[1, 5, 10],
    item_count=train_dataset.item_count,
    postprocessors=[RemoveSeenItems(sequential_validation_dataset)]
)

# csv_logger = CSVLogger(save_dir=".logs/train", name="Bert4Rec_example")
wandb_logger = WandbLogger(project="Bert4Rec")

trainer = L.Trainer(
    max_epochs=100,
    callbacks=[checkpoint_callback_bert, validation_metrics_callback],
    logger=wandb_logger,
)

train_dataloader = DataLoader(
    dataset=Bert4RecTrainingDataset(
        sequential_train_dataset,
        max_sequence_length=MAX_SEQ_LEN,
    ),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

validation_dataloader = DataLoader(
    dataset=Bert4RecValidationDataset(
        sequential_validation_dataset,
        sequential_validation_gt,
        sequential_train_dataset,
        max_sequence_length=MAX_SEQ_LEN,
    ),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

trainer.fit(
    model,
    train_dataloaders=train_dataloader,
    val_dataloaders=validation_dataloader,
)

wandb.finish()

INFO: GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs


INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
  | Name   | Type             | Params | Mode 
----------------------------------------------------
0 | _model | Bert4RecModel    | 4.6 M  | train
1 | _loss  | CrossEntropyLoss | 0      | train
----------------------------------------------------
4.6 M     Trainable params
0         Non-trainable params
4.6 M     Total params
18.247    Total estimated model params size (MB)
38        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/opt/conda/lib/python3.10/site-packages/lightning/pytorch/loops/fit_loop.py:298: The number of training batches (12) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training: |          | 0/? [00:00<?, ?it/s]

...

k                 1        10         5
mrr        0.040232  0.082613  0.071918
ndcg       0.040232  0.113434  0.087237
precision  0.040232  0.021573  0.026821
recall     0.040232  0.215728  0.134106



Validation: |          | 0/? [00:00<?, ?it/s]

INFO: Epoch 99, global step 1200: 'recall@10' was not in top 1
INFO: `Trainer.fit` stopped: `max_epochs=100` reached.


k                 1        10         5
mrr        0.040397  0.082897  0.071943
ndcg       0.040397  0.114126  0.087328
precision  0.040397  0.021788  0.026887
recall     0.040397  0.217881  0.134437



epoch,▁▁▁▂▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇█████
mrr@1,▁▁▁▁▂▂▂▂▂▂▄▅▅▄▅▆▆▅▆▅▆▇▇▇▇▇▇▇▇▇█▇█▆█▇█▇▇█
mrr@10,▁▁▁▁▁▂▂▂▂▂▂▄▅▅▅▆▆▆▇▇▇▇▇▇▇▇▇█▇█▇█▇███████
mrr@5,▁▁▁▁▁▂▂▂▂▂▃▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇█▇██▇█████
ndcg@1,▁▁▁▁▁▂▂▂▂▂▅▅▅▆▆▅▆▇▇▇▇▇▇▇▇▇▇▇█▇▆▇▇██▇██▇█
ndcg@10,▁▁▁▁▂▂▂▂▃▄▅▆▆▆▆▆▆▆▇▆▇▇▇▇▇▇▇██▇█▇████████
ndcg@5,▁▁▁▁▂▂▂▂▂▂▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇█▇▇█▇█████████
precision@1,▁▁▁▂▂▁▂▂▂▂▃▄▅▄▅▅▆▇▆▆▆▇▇▇▇▇▇▇▇█▇█▇▇▇▇▇▇█▇
precision@10,▁▁▁▁▁▂▂▂▂▂▄▅▅▆▆▆▇▇▇▇▇▇▇███▇▇█▇██▇██▇████
precision@5,▁▁▁▁▁▂▁▁▂▂▃▄▅▅▅▆▆▆▆▇▇▇▇▇█▇██▇▇▇█████████
recall@1,▁▁▁▁▁▂▂▁▂▂▄▅▅▅▅▆▆▇▆▇▆▇▇▇▇▇▇▇█▇▆▇▇▇▇▇▇▇▇▇


In [72]:
wandb.finish()

## Инициализация и обучение SASRec

In [43]:
MAX_SEQ_LEN = 200
BATCH_SIZE = 512
NUM_WORKERS = 4

model = SasRec(
    tensor_schema,
    block_count=2,
    head_count=2,
    max_seq_len=MAX_SEQ_LEN,
    hidden_size=300,
    dropout_rate=0.5,
    optimizer_factory=FatOptimizerFactory(learning_rate=0.001),
)

checkpoint_callback = ModelCheckpoint(
    dirpath=".checkpoints",
    save_top_k=1,
    verbose=True,
    # if you use multiple dataloaders, then add the serial number of the dataloader to the suffix of the metric name.
    # For example,"recall@10/dataloader_idx_0"
    monitor="recall@10",
    mode="max",
)

validation_metrics_callback = ValidationMetricsCallback(
    metrics=["mrr", "ndcg", "recall", "precision"],
    ks=[1, 5, 10],
    item_count=train_dataset.item_count,
    postprocessors=[RemoveSeenItems(sequential_validation_dataset)]
)

wandb_logger = WandbLogger(project="Bert4Rec")

trainer = L.Trainer(
    max_epochs=100,
    callbacks=[checkpoint_callback, validation_metrics_callback],
    logger=wandb_logger,
)

train_dataloader = DataLoader(
    dataset=SasRecTrainingDataset(
        sequential_train_dataset,
        max_sequence_length=MAX_SEQ_LEN,
    ),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

validation_dataloader = DataLoader(
    dataset=SasRecValidationDataset(
        sequential_validation_dataset,
        sequential_validation_gt,
        sequential_train_dataset,
        max_sequence_length=MAX_SEQ_LEN,
    ),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

trainer.fit(
    model,
    train_dataloaders=train_dataloader,
    val_dataloaders=validation_dataloader,
)

wandb.finish()

INFO: GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs


/opt/conda/lib/python3.10/site-packages/lightning/pytorch/callbacks/model_checkpoint.py:654: Checkpoint directory /kaggle/working/.checkpoints exists and is not empty.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO: 
  | Name   | Type             | Params | Mode 
----------------------------------------------------
0 | _model | SasRecModel      | 2.3 M  | train
1 | _loss  | CrossEntropyLoss | 0      | train
----------------------------------------------------
2.3 M     Trainable params
0         Non-trainable params
2.3 M     Total params
9.247     Total estimated model params size (MB)
35        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/opt/conda/lib/python3.10/site-packages/lightning/pytorch/loops/fit_loop.py:298: The number of training batches (12) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training: |          | 0/? [00:00<?, ?it/s]

...

k                 1        10         5
mrr        0.069536  0.133617  0.119434
ndcg       0.069536  0.177390  0.142672
precision  0.069536  0.032152  0.042682
recall     0.069536  0.321523  0.213411



Validation: |          | 0/? [00:00<?, ?it/s]

INFO: Epoch 99, global step 1200: 'recall@10' was not in top 1
INFO: `Trainer.fit` stopped: `max_epochs=100` reached.


k                 1        10         5
mrr        0.068709  0.133209  0.120069
ndcg       0.068709  0.176504  0.144469
precision  0.068709  0.031854  0.043808
recall     0.068709  0.318543  0.219040



epoch,▁▁▁▁▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇██
mrr@1,▁▁▁▁▁▄▅▅▆▇▇▇▇▇▇▇█▇▇▇██▇▇▇▇▇▇█▇▇█████████
mrr@10,▁▁▂▃▄▆▆▇▇▇▇▇▇▇▇▇████████████████████████
mrr@5,▁▁▁▃▄▅▆▇▇▇▇▇▇▇▇█▇███████████████████████
ndcg@1,▁▁▁▅▆▆▇▇▇▇▇▇▇▇▇▇▇▇█▇▇█▇█▇█▇▇███▇████████
ndcg@10,▁▁▁▄▅▇▇▇▇▇▇▇▇▇██████████████████████████
ndcg@5,▁▃▆▇▇▇▇▇▇▇▇▇▇███████████████████████████
precision@1,▁▁▃▆▇▇▇▇▇▇█▇▇▇█████▇▇█████▇█▇███████████
precision@10,▁▁▁▁▁▄▅▆▆▆▆▇▇▇▇▇▇███████████████████████
precision@5,▁▁▂▄▄▆▆▆▆▇▇▇▇▇▇▇████████████████████████
recall@1,▁▂▃▅▅▆▇▇▇▇▇▇▇▇▇▇█▇█▇▇▇▇▇█▇▇▇█▇█▇█▇██▇███


## Сохранение лучших моделей

In [55]:
best_model_rec = Bert4Rec.load_from_checkpoint(checkpoint_callback_bert.best_model_path)
best_model_sas = SasRec.load_from_checkpoint(checkpoint_callback.best_model_path)

## Запуск моделей и получение последовательностей рекомендаций

In [61]:
prediction_dataloader_rec = DataLoader(
    dataset=Bert4RecPredictionDataset(
        sequential_test_dataset,
        max_sequence_length=MAX_SEQ_LEN,
    ),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

prediction_dataloader_sas = DataLoader(
    dataset=SasRecPredictionDataset(
        sequential_test_dataset,
        max_sequence_length=MAX_SEQ_LEN,
    ),
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)

# csv_logger = CSVLogger(save_dir=".logs/test", name="Bert4Rec_example")
wandb_logger = WandbLogger(project="Bert4Rec")

TOPK = [1, 10, 20, 100]

postprocessors = [RemoveSeenItems(sequential_test_dataset)]

pandas_prediction_callback = PandasPredictionCallback(
    top_k=max(TOPK),
    query_column="user_id",
    item_column="item_id",
    rating_column="score",
    postprocessors=postprocessors,
)


trainer = L.Trainer(
    callbacks=[
        pandas_prediction_callback
    ], 
    logger=wandb_logger, 
    inference_mode=True
)
trainer.predict(best_model_rec, dataloaders=prediction_dataloader_rec, return_predictions=False)
pandas_res_rec = pandas_prediction_callback.get_result()
trainer.predict(best_model_sas, dataloaders=prediction_dataloader_sas, return_predictions=False)
pandas_res_sas = pandas_prediction_callback.get_result()

recommendations_rec = tokenizer.query_and_item_id_encoder.inverse_transform(pandas_res_rec)
recommendations_sas = tokenizer.query_and_item_id_encoder.inverse_transform(pandas_res_sas)

## Подсчет оффлайн-метрик

In [68]:
init_args = {"query_column": "user_id", "rating_column": "score"}

result_metrics_rec = OfflineMetrics(
    [Recall(TOPK), Precision(TOPK), NDCG(TOPK), MRR(TOPK)], **init_args
)(recommendations_rec, raw_test_gt)
result_metrics_sas = OfflineMetrics(
    [Recall(TOPK), Precision(TOPK), NDCG(TOPK), MRR(TOPK)], **init_args
)(recommendations_sas, raw_test_gt)

metrics_to_df(result_metrics_rec)
metrics_to_df(result_metrics_sas)